# Single-Runner SAC Trader (stream tapes)

The agent **observes the whole race** but **trades one runner per episode** (1st, 2nd, ... nth favourite); every
runner of every race is its own episode. It starts with $100, can back and lay any size at any ladder price every 2 s,
and is **greened up without bias at the off**.

Data and fills are the `ahr_rl` research stack: stream recordings compiled into 0.5 s tapes (8-level ladders, every
trade, scratchings, base-rate commission) and the conservative exchange simulator (latency, queue position, fills from
recorded trades), plus Betfair-style cross-matching. Code: `single_runner/` (see `single_runner/README.md`).

In [ ]:
### CELL 1 - DRIVE + CODE ###
from google.colab import drive
drive.mount('/content/drive')

!git clone -q -b claude/pensive-bell-6dmlvb https://github.com/tristanhowells/AHR_RL.git /content/AHR_RL 2>/dev/null || git -C /content/AHR_RL pull -q
%cd /content/AHR_RL
!pip -q install orjson "gymnasium>=1.0" pyarrow
!python -m pytest -q tests/test_single_runner.py tests/test_exchange.py

In [ ]:
### CELL 2 - TAPES (shared with the research notebook; compiling is incremental) ###
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
TAPES = f'{DRIVE}/rl_tapes_v1'
NCPU = os.cpu_count()
os.makedirs(TAPES, exist_ok=True)
!python -m ahr_rl.tape "{DRIVE}/recordings" "{TAPES}" --workers {NCPU}
!python -m ahr_rl.catalogue "{TAPES}" "{DRIVE}/catalogues"     # attach form features (skips if done)
!mkdir -p /content/tapes && rsync -a "{TAPES}/" /content/tapes/  # local copy reads much faster
!ls /content/tapes | wc -l

In [ ]:
### CELL 3 - CONFIG ###
OUT_DIR = f'{DRIVE}/rl_runs_single_runner/sac_v2'

RANKS = None             # e.g. (1, 2, 3) for the top-3 favourites only; None = every runner
RACE_TYPE = 'all'        # 'flat' = thoroughbreds only (the race study found harness untradeable pre-off)
METRO_ONLY = False
TOTAL_STEPS = 1_000_000  # agent decisions
EVAL_EVERY = 50_000
MAX_EVAL_EPISODES = 2000

from single_runner import EnvConfig, SACConfig
env_cfg = EnvConfig(
    initial_balance=100.0,
    min_stake=5.0,           # Betfair AU minimum for opening orders
    tick_range=10,           # price action spans +/- 10 ladder ticks around the touch
    decision_every=4,        # tape steps (0.5 s) per decision -> 2 s
    fill_mode='realistic',   # 'no_queue' / 'touch' are optimistic sensitivity checks only
    cross_matching=True,
    greenup_mode='fair',     # 'fair' (unbiased) | 'cross' (pays the spread)
    reward_scale=10.0,
    random_start_s=0.0,
    runner_ranks=RANKS,
)
sac_cfg = SACConfig(buffer_size=300_000, batch_size=256, lr=3e-4, gamma=0.99,
                    learning_starts=5_000, updates_per_step=1)

In [ ]:
### CELL 4 - SPLIT + BASELINES ###
import argparse
import numpy as np
from ahr_rl import race_filter
from ahr_rl.env import list_tapes, split_by_date
from single_runner import SingleRunnerTradingEnv, episode_specs
from single_runner.train import evaluate, summarise

flt = argparse.Namespace(race_type=RACE_TYPE, metro_only=METRO_ONLY,
                         catalogues=f'{DRIVE}/catalogues', races_csv=None)
tapes = race_filter.filter_paths(list_tapes('/content/tapes/*.npz'), flt)
train_tapes, val_tapes, test_tapes = split_by_date(tapes, 0.15, 0.15)
print(f'{len(tapes)} races -> train {len(train_tapes)} / val {len(val_tapes)} / test {len(test_tapes)}')

class _Const:
    def __init__(self, a): self.a = np.asarray(a, np.float32)
    def act(self, obs, deterministic=True): return self.a
class _Random:
    def act(self, obs, deterministic=True): return np.random.uniform(-1, 1, 6).astype(np.float32)

probe_env = SingleRunnerTradingEnv(val_tapes, env_cfg)
probe = episode_specs(probe_env, val_tapes[:20])
for name, pol in [('do-nothing', _Const(-np.ones(6))), ('random', _Random())]:
    s = summarise(evaluate(pol, probe_env, probe))
    print(f"{name:11s} mean pnl {s['mean_pnl']:+.2f}  t(race) {s['t_race']:+.2f}  matched {s['mean_matched']:.0f}")

In [ ]:
### CELL 5 - TRAIN ###
from single_runner.train import train
agent = train(train_tapes, val_tapes, OUT_DIR, env_cfg, sac_cfg,
              total_steps=TOTAL_STEPS, eval_every=EVAL_EVERY,
              max_eval_episodes=MAX_EVAL_EPISODES, seed=42)

## Score once on TEST
Run this **once**, after you've picked the checkpoint on validation. `fair` is the unbiased green-up the agent was
trained on; `cross` re-scores the same policy with a spread-paying green-up. Research gate: mean > 0 with t > 2.

In [ ]:
### CELL 6 - TEST ###
import pandas as pd
from single_runner.train import evaluate_checkpoint
res, extra = evaluate_checkpoint(f'{OUT_DIR}/best.pt', test_tapes, env_cfg, f'{OUT_DIR}/test_episodes.csv')
print('checkpoint step', extra.get('step'))
for mode, s in res.items():
    gate = 'PASS' if s['mean_pnl'] > 0 and s['t_race'] > 2 else 'fail'
    print(f"{mode:5s} mean {s['mean_pnl']:+.3f}  t(race) {s['t_race']:+.2f}  win {s['win_rate']:.2f}  "
          f"trade {s['trade_rate']:.2f}  gate {gate}")
ep = pd.read_csv(f'{OUT_DIR}/test_episodes.csv')
print(ep.groupby(['greenup', 'target_rank']).agg(n=('pnl', 'size'), mean_pnl=('pnl', 'mean'),
                                                 win=('pnl', lambda x: (x > 0).mean()),
                                                 matched=('back_matched', 'mean')).round(3))